In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("mini-project")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-db8a21e0-e96c-4816-b765-0c1eb566bb41;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 144ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

In [2]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

products_schema = StructType([
    StructField("product_id",     StringType(), True),
    StructField("product_name",   StringType(), True),
    StructField("category",       StringType(), True),
    StructField("sub_category",   StringType(), True),
    StructField("unit_price",     DoubleType(), True),
    StructField("cost_price",     DoubleType(), True),
    StructField("supplier",       StringType(), True),
    StructField("stock_quantity", IntegerType(),True)
])
BUCKET="pyspark-30-days-rahul-2026"
orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
products_df  = spark.read.option("header","true").schema(products_schema).csv(f"s3a://{BUCKET}/data/products.csv")
print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()} | Products: {products_df.count()}")

26/08/06 09:37:05 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Orders: 100 | Customers: 25 | Products: 20


***Clean the data (drop duplicate orders)***

In [3]:
orders_df=orders_df.dropDuplicates(subset=["order_id"])
orders_df.show(5)

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|          15|Delivered|   Credit Card|Midwest|
|   O0004|       C004|      P006|2023-01-12|       2|     89.99|           5|Delivered|    Debit Card|  South|
|   O0005|       C005|      P002|2023-01-15|       3|     29.99|           0|Delivered|   Credit Card|   West|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
o

***Join orders with customers***

In [4]:
orders_with_customers = orders_df.join(customers_df, on="customer_id", how="inner")
orders_with_customers.show(5)

+-----------+--------+----------+----------+--------+----------+------------+---------+--------------+-------+----------+---------+--------------------+-----------+-----+-------+-----------+----------+
|customer_id|order_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|first_name|last_name|               email|       city|state|country|signup_date|   segment|
+-----------+--------+----------+----------+--------+----------+------------+---------+--------------+-------+----------+---------+--------------------+-----------+-----+-------+-----------+----------+
|       C001|   O0001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|     James| Anderson|james.anderson@em...|   New York|   NY|    USA| 2021-03-15|Enterprise|
|       C002|   O0002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|     Maria|   Garcia|maria.garcia@emai...|Los Angeles|   CA|    USA| 2021-05-22|  

***Compute revenue per order***

In [5]:
orders_revenue=orders_with_customers.withColumn(
    'revenue',F.round(F.col('unit_price') * F.col('quantity') * (1 - F.col('discount_pct') / 100), 2)
)
orders_revenue.show(5,truncate=False)

+-----------+--------+----------+----------+--------+----------+------------+---------+--------------+-------+----------+---------+------------------------+-----------+-----+-------+-----------+----------+-------+
|customer_id|order_id|product_id|order_date|quantity|unit_price|discount_pct|status   |payment_method|region |first_name|last_name|email                   |city       |state|country|signup_date|segment   |revenue|
+-----------+--------+----------+----------+--------+----------+------------+---------+--------------+-------+----------+---------+------------------------+-----------+-----+-------+-----------+----------+-------+
|C001       |O0001   |P001      |2023-01-05|2       |1299.99   |10          |Delivered|Credit Card   |East   |James     |Anderson |james.anderson@email.com|New York   |NY   |USA    |2021-03-15 |Enterprise|2339.98|
|C002       |O0002   |P005      |2023-01-07|1       |449.99    |0           |Delivered|PayPal        |West   |Maria     |Garcia   |maria.garcia@

***Group by customer and aggregate***

In [6]:
from pyspark.sql import functions as F

customer_summary = (
    orders_revenue
    .groupBy("customer_id", "first_name", "last_name", "segment","region")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_spend"),
        F.round(F.avg("revenue"), 2).alias("avg_spend")
    )
    .orderBy(F.desc("total_spend"))
)

customer_summary.show(5, truncate=False)

+-----------+----------+---------+----------+-------+-----------+---------+
|customer_id|first_name|last_name|segment   |region |total_spend|avg_spend|
+-----------+----------+---------+----------+-------+-----------+---------+
|C001       |James     |Anderson |Enterprise|East   |3449.93    |689.99   |
|C003       |Robert    |Johnson  |Enterprise|Midwest|2687.77    |537.55   |
|C017       |Charles   |Robinson |Enterprise|West   |2189.89    |547.47   |
|C002       |Maria     |Garcia   |SMB       |West   |2076.42    |415.28   |
|C006       |Patricia  |Davis    |Enterprise|East   |2039.92    |509.98   |
+-----------+----------+---------+----------+-------+-----------+---------+
only showing top 5 rows


***Find each customer's most expensive order using a window function***

In [7]:
top_order_window = Window.partitionBy("customer_id").orderBy(F.col("revenue").desc())

top_orders = orders_revenue.withColumn(
    "row_num", F.row_number().over(top_order_window)
).filter(F.col("row_num") == 1).select(
    F.col("customer_id"),
    F.col("order_id").alias("top_order_id"),
    F.col("revenue").alias("top_order_revenue"))
top_orders.show(5, truncate=False)

+-----------+------------+-----------------+
|customer_id|top_order_id|top_order_revenue|
+-----------+------------+-----------------+
|C001       |O0001       |2339.98          |
|C002       |O0072       |1169.99          |
|C003       |O0003       |1189.97          |
|C004       |O0024       |1169.99          |
|C005       |O0100       |764.98           |
+-----------+------------+-----------------+
only showing top 5 rows


***Join aggregated summary with top order and customer details***

In [8]:
final_summary = customer_summary.join(top_orders, on="customer_id", how="left")
final_summary.orderBy(F.col("total_spend").desc()).show(10)

+-----------+----------+---------+----------+-------+-----------+---------+------------+-----------------+
|customer_id|first_name|last_name|   segment| region|total_spend|avg_spend|top_order_id|top_order_revenue|
+-----------+----------+---------+----------+-------+-----------+---------+------------+-----------------+
|       C001|     James| Anderson|Enterprise|   East|    3449.93|   689.99|       O0001|          2339.98|
|       C003|    Robert|  Johnson|Enterprise|Midwest|    2687.77|   537.55|       O0003|          1189.97|
|       C017|   Charles| Robinson|Enterprise|   West|    2189.89|   547.47|       O0062|           899.98|
|       C002|     Maria|   Garcia|       SMB|   West|    2076.42|   415.28|       O0072|          1169.99|
|       C006|  Patricia|    Davis|Enterprise|   East|    2039.92|   509.98|       O0051|          1039.99|
|       C010|  Jennifer|  Jackson|       SMB|   West|    2034.95|   508.74|       O0080|          1104.99|
|       C024|   Dorothy|    Allen|   

In [9]:
final_summary.write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet(f"s3a://{BUCKET}/output/customer_revenue_summary/")

# Verify
verify_df = spark.read.parquet(f"s3a://{BUCKET}/output/customer_revenue_summary/")
print(f"Rows written: {final_summary.count()}")
print(f"Rows read back: {verify_df.count()}")
verify_df.show(5)

Rows written: 25


Rows read back: 25


+-----------+----------+---------+-------+-----------+---------+------------+-----------------+------+
|customer_id|first_name|last_name|segment|total_spend|avg_spend|top_order_id|top_order_revenue|region|
+-----------+----------+---------+-------+-----------+---------+------------+-----------------+------+
|       C002|     Maria|   Garcia|    SMB|    2076.42|   415.28|       O0072|          1169.99|  West|
|       C005|   Michael|    Brown|Startup|     1663.9|   332.78|       O0100|           764.98|  West|
|       C022|     Betty|   Walker|    SMB|    1039.96|   346.65|       O0042|           899.98|  West|
|       C013|    Joseph|   Martin|    SMB|     699.91|   174.98|       O0058|           219.98|  West|
|       C008|   Barbara|   Taylor|Startup|     889.91|   222.48|       O0078|           629.99|  West|
+-----------+----------+---------+-------+-----------+---------+------------+-----------------+------+
only showing top 5 rows


***Deliverable 3***

Which customer segment has the highest average order value?

Which region has the most customers in the top-spender tier (top 5 by total_spend)?


In [10]:
segment_avg = (
    orders_revenue
    .groupBy("segment")
    .agg(
        F.round(F.avg("revenue"), 2).alias("avg_order_value")
    )
    .orderBy(F.desc("avg_order_value"))
)

segment_avg.show()

+----------+---------------+
|   segment|avg_order_value|
+----------+---------------+
|Enterprise|         458.85|
|       SMB|         376.74|
|   Startup|         369.79|
+----------+---------------+



In [12]:
import plotly.express as px

pdf = segment_avg.toPandas()

fig = px.bar(
    pdf,
    x="segment",
    y="avg_order_value",
    color="avg_order_value",
    text="avg_order_value",
    color_continuous_scale="Viridis",
    title="Average Order Value by Customer Segment"
)

fig.update_traces(
    texttemplate="$%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Customer Segment",
    yaxis_title="Average Order Value",
    title_x=0.5,
    showlegend=False,
    height=500
)

fig.show()

In [13]:
top5_customers = (
    customer_summary
    .limit(5)
)

region_top_spenders = (
    top5_customers
    .groupBy("region")
    .count()
    .withColumnRenamed("count", "top_spenders")
    .orderBy(F.desc("top_spenders"))
)

region_top_spenders.show()

+-------+------------+
| region|top_spenders|
+-------+------------+
|   East|           2|
|   West|           2|
|Midwest|           1|
+-------+------------+



In [15]:
fig = px.pie(
    pdf,
    names="region",
    values="top_spenders",
    hole=0.45,
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="Share of Top-Spending Customers by Region"
)

fig.update_traces(textinfo="percent+label")
fig.update_layout(title_x=0.5)

fig.show()